# Ejercicios: complejidad aleatoria y tests probabilísticos

La complejidad aleatoria estudia qué problemas se pueden resolver eficientemente usando monedas.
En este cuaderno implementarás y verificarás dos algoritmos canónicos de clase co-RP:
el test de Miller-Rabin para primalidad y la verificación de multiplicación matricial (Freivalds).

**Artículo asociado:** [Complejidad aleatoria](../../04-complejidad-computacional/08-complejidad-aleatoria.md)

In [ ]:
import random
import math

def verificar_casos(nombre, funcion, casos):
    todos_ok = True
    for entrada, esperado in casos:
        if isinstance(entrada, tuple):
            resultado = funcion(*entrada)
        else:
            resultado = funcion(entrada)
        ok = '✓' if resultado == esperado else '✗'
        print(f'  {ok} entrada={str(entrada):20} esperado={esperado} obtenido={resultado}')
        if resultado != esperado:
            todos_ok = False
    print(f'{nombre}: ' + ('todos correctos.' if todos_ok else 'hay errores.'))

print('Utilidades cargadas.')

## Ejercicio 1: test de Miller-Rabin

El test de Miller-Rabin es un algoritmo probabilístico en **co-RP** para primalidad:
- Si $n$ es primo: siempre devuelve `True`.
- Si $n$ es compuesto: devuelve `False` con probabilidad $\geq 3/4$ por iteración.

**Fundamento:** para $n-1 = 2^s \cdot d$ con $d$ impar, $n$ es **primo fuerte en base $a$** si:
- $a^d \equiv 1 \pmod{n}$, o
- $a^{2^r d} \equiv -1 \pmod{n}$ para algún $r \in \{0,\ldots,s-1\}$.

Si $n$ es compuesto, a lo sumo $1/4$ de las bases $a$ son testigos erróneos.

**Tarea:** completa la función `miller_rabin_test(n, a)` que devuelve `True` si $n$ pasa
el test en base $a$ (es decir, si $n$ es primo fuerte en base $a$).

In [ ]:
def miller_rabin_test(n, a):
    """
    Devuelve True si n pasa el test de Miller-Rabin con testigo a.
    n debe ser impar y >= 3.
    """
    if n % 2 == 0:
        return n == 2

    # Escribir n-1 = 2^s * d con d impar
    s, d = 0, n - 1
    while d % 2 == 0:
        s += 1
        d //= 2

    # TODO: Implementar el test
    # 1. Calcular x = pow(a, d, n)
    # 2. Si x == 1 o x == n-1: devolver True
    # 3. Repetir s-1 veces: x = pow(x, 2, n). Si x == n-1: devolver True
    # 4. Devolver False
    pass  # eliminar cuando se complete


def es_primo_mr(n, k=40, seed=None):
    """
    Test de Miller-Rabin con k testigos aleatorios.
    Probabilidad de error <= 4^(-k).
    """
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False
    rng = random.Random(seed)
    for _ in range(k):
        a = rng.randint(2, n - 2)
        if not miller_rabin_test(n, a):
            return False
    return True


# Descomenta cuando hayas completado miller_rabin_test:
# primos_conocidos = [2, 3, 5, 7, 11, 13, 17, 19, 23, 97, 101, 1009, 7919]
# compuestos = [4, 6, 9, 15, 25, 100, 1001, 8000, 9999]
#
# print('Test en primos conocidos (todos deben ser True):')
# for p in primos_conocidos:
#     resultado = es_primo_mr(p, seed=42)
#     print(f'  {"✓" if resultado else "✗"}  {p}: {resultado}')
#
# print()
# print('Test en compuestos (todos deben ser False):')
# for c in compuestos:
#     resultado = es_primo_mr(c, seed=42)
#     print(f'  {"✓" if not resultado else "✗"}  {c}: {resultado}')

### Solución de referencia para el ejercicio 1

In [ ]:
def miller_rabin_test_ref(n, a):
    if n % 2 == 0:
        return n == 2
    s, d = 0, n - 1
    while d % 2 == 0:
        s += 1
        d //= 2
    x = pow(a, d, n)
    if x == 1 or x == n - 1:
        return True
    for _ in range(s - 1):
        x = pow(x, 2, n)
        if x == n - 1:
            return True
    return False


def es_primo_mr_ref(n, k=40, seed=None):
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False
    rng = random.Random(seed)
    for _ in range(k):
        a = rng.randint(2, n - 2)
        if not miller_rabin_test_ref(n, a):
            return False
    return True


primos_conocidos = [2, 3, 5, 7, 11, 13, 17, 19, 23, 97, 101, 1009, 7919]
compuestos = [4, 6, 9, 15, 25, 100, 1001, 8000, 9999]

print('Test en primos (deben ser True):')
for p in primos_conocidos:
    resultado = es_primo_mr_ref(p, seed=42)
    print(f'  {"✓" if resultado else "✗"}  {p}: {resultado}')

print()
print('Test en compuestos (deben ser False):')
for c in compuestos:
    resultado = es_primo_mr_ref(c, seed=42)
    print(f'  {"✓" if not resultado else "✗"}  {c}: {resultado}')

## Ejercicio 2: algoritmo de Freivalds (verificación matricial)

Dados tres matrices $A$, $B$, $C$ de $n \times n$, comprobar si $AB = C$ requiere $O(n^3)$ pasos
calculando $AB$ explícitamente. El algoritmo de **Freivalds** verifica esto en $O(n^2)$
con probabilidad de error $\leq 1/2$ por iteración:

1. Elegir un vector aleatorio $r \in \{0,1\}^n$.
2. Calcular $ABr$ (primero $Br$, luego $A(Br)$) en $O(n^2)$.
3. Calcular $Cr$ en $O(n^2)$.
4. Si $ABr \neq Cr$: devolver `False`. Si son iguales: continuar.

**Propiedad:** si $AB \neq C$, $\Pr[ABr = Cr] \leq 1/2$.

**Tarea:** completa la función `freivalds_test(A, B, C, k, seed)` que realiza $k$ rondas.

In [ ]:
def mat_vec(M, v):
    """Multiplica una matriz M por un vector v (listas de listas)."""
    n = len(M)
    return [sum(M[i][j] * v[j] for j in range(n)) for i in range(n)]


def freivalds_test(A, B, C, k=20, seed=None):
    """
    Devuelve True si AB probablemente = C (probabilidad de error <= 2^(-k)).
    Devuelve False si seguro AB != C.
    """
    n = len(A)
    rng = random.Random(seed)
    for _ in range(k):
        # TODO: generar r aleatorio en {0,1}^n y verificar ABr == Cr
        pass
    return True  # si todas las rondas pasan


# Descomenta cuando hayas completado freivalds_test:
# A = [[1, 2], [3, 4]]
# B = [[5, 6], [7, 8]]
# C_correcto = [[19, 22], [43, 50]]    # AB correcto
# C_incorrecto = [[19, 22], [43, 51]]  # error en una entrada
#
# print('AB = C_correcto:', freivalds_test(A, B, C_correcto, seed=42))
# print('AB = C_incorrecto:', freivalds_test(A, B, C_incorrecto, seed=42))

### Solución de referencia para el ejercicio 2

In [ ]:
def freivalds_test_ref(A, B, C, k=20, seed=None):
    n = len(A)
    rng = random.Random(seed)
    for _ in range(k):
        r = [rng.randint(0, 1) for _ in range(n)]
        Br = mat_vec(B, r)
        ABr = mat_vec(A, Br)
        Cr = mat_vec(C, r)
        if ABr != Cr:
            return False
    return True


A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
C_correcto   = [[19, 22], [43, 50]]
C_incorrecto = [[19, 22], [43, 51]]

print('Verificación de Freivalds:')
print(f'  AB = C_correcto:   {freivalds_test_ref(A, B, C_correcto, seed=42)}   (esperado: True)')
print(f'  AB = C_incorrecto: {freivalds_test_ref(A, B, C_incorrecto, seed=42)}  (esperado: False)')

# Estudio empírico del error
print()
print('Probabilidad empírica de error de Freivalds con k=1 (debe ser ~0.5):')
rng = random.Random(0)
errores = sum(
    freivalds_test_ref(A, B, C_incorrecto, k=1, seed=rng.randint(0, 10**6))
    for _ in range(1000)
)
print(f'  Falsos positivos en 1000 pruebas: {errores} ({errores/10:.1f}%)')

## Ejercicio 3: amplificación de éxito

Si un algoritmo tiene probabilidad de error $\leq 1/2$ por ronda, repetirlo $k$ veces
reduce el error a $\leq 2^{-k}$.

Verifica esto empíricamente con el test de Freivalds: mide la fracción de falsos positivos
para distintos valores de $k$.

In [ ]:
print('Amplificación del test de Freivalds:')
print(f'{"k (rondas)":>12}  {"Falsos positivos / 1000":>25}  {"Cota 2^(-k)":>14}')
for k in [1, 2, 3, 5, 10, 20]:
    rng = random.Random(0)
    falsos = sum(
        freivalds_test_ref(A, B, C_incorrecto, k=k, seed=rng.randint(0, 10**9))
        for _ in range(1000)
    )
    cota = 2 ** (-k)
    print(f'{k:>12}  {falsos:>25}  {cota:>14.6f}')

print()
print('Con k=20, la probabilidad de error es < 10^-6: prácticamente cero fallos en 1000 pruebas.')

## Reflexión

- Miller-Rabin y Freivalds son ejemplos de algoritmos en **co-RP**: nunca dan falsos negativos
  (si la respuesta es verdadera, siempre lo detectan), pero pueden dar falsos positivos con
  probabilidad pequeña.
- La **amplificación** permite reducir el error exponencialmente con rondas adicionales,
  todas de coste polinomial.
- Miller-Rabin usa $O(k \log^2 n)$ operaciones de módulo; Freivalds usa $O(k n^2)$.
  Ambos son polinomiales en el tamaño de entrada.
- **Pregunta:** ¿qué significa que un algoritmo esté en ZPP? ¿Puedes construir un algoritmo
  ZPP para primalidad a partir de Miller-Rabin?

In [ ]:
# ── Verificación automática ──────────────────────────────────────────────────
import random

def miller_rabin(n, k=10):
    """Test de primalidad de Miller-Rabin."""
    if n < 2: return False
    if n == 2 or n == 3: return True
    if n % 2 == 0: return False
    r, d = 0, n - 1
    while d % 2 == 0: r += 1; d //= 2
    for _ in range(k):
        a = random.randint(2, n - 2)
        x = pow(a, d, n)
        if x == 1 or x == n - 1: continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1: break
        else: return False
    return True

primos = [2, 3, 5, 7, 11, 13, 17, 97, 101, 9973]
compuestos = [4, 9, 15, 25, 100, 9975]
for p in primos:
    assert miller_rabin(p, k=20), f"{p} es primo pero Miller-Rabin lo rechaza"
for c in compuestos:
    assert not miller_rabin(c, k=20), f"{c} es compuesto pero Miller-Rabin lo acepta"
print("✓ Miller-Rabin pasa todos los casos de prueba")